# Lab 03: Descriptive Statistics & Aggregation

Your Name(s): Kristian Janci, JT HUFF

**Context:** We are analyzing a month of transaction data from a campus coffee shop. 

**Dataset Variables:** 
* `transaction_id`: Unique ID for the purchase
* `day_of_week`: Monday - Sunday
* `time_of_day`: Morning, Afternoon, Evening
* `item_type`: Coffee, Tea, Pastry, Sandwich, Merchandise
* `quantity`: Number of items purchased in the transaction
* `amount`: Total cost of the transaction
* `student_discount`: Boolean (True/False)


## Part 1: Descriptive Statistics

Before slicing the data into groups, we need to understand the overall properties of our dataset.



### 1. PREDICT
Think about a typical coffee shop. Explain your thought process for each of the questions below.
* Do you expect the *mean* (average) transaction amount to be higher or lower than the *median* transaction amount? Why? 
* What do you think the most expensive item type in the shop is?

We think the mean and the median will be close to each other because all items in a coffee shop are similarly priced and most sales are probably in the middle range.
The most expensive items are usually the large iced lattes (special edition)



### 2. RUN
Execute the cell below to generate the descriptive statistics for the `amount` column.

In [1]:
# read in data
import pandas as pd
df = pd.read_csv('campus_coffee.csv')

df['amount'].describe()

count    10000.000000
mean         7.981092
std          9.058046
min          2.230000
25%          3.910000
50%          4.780000
75%          9.152500
max        213.330000
Name: amount, dtype: float64

### 3. EXPLAIN
Look at the output above. 
* Was your prediction about the mean vs. median correct? 
* The `max` amount is much higher than the 75th percentile (`75%`). In the context of our coffee shop, what real-world purchases might explain this long right tail?

No we thought they're going to be similar but the mean endeded up being much higher, probably because of the outlier at 213 which skewed the mean upwards.
In a normal coffee shop, most people buy one coffee and maybe a bagel or desert and then rarely someone comes in to buy for a larger group, which explains the long right tail.
People can't go below $0 so there's no left tail.



## Part 2: Aggregation (Split-Apply-Combine)

Now, we want to compare different categories. Let's look at purchasing behavior based on the time of day.

### 1. PREDICT
* Which `time_of_day` (Morning, Afternoon, or Evening) do you expect to have the highest *total number of sales* (count)? 
* Which `time_of_day` do you expect to have the highest *average transaction price* (mean)? 

We think the most sales are in the morning because caffeine is usually consumed when people wake up to get more energy.

We think the afternoon has the highest average transaction price because we think that in the morning people just get coffee (they're in a rush), in the evening people might buy more snacks and less coffee, but in the afternoon they might get both.



### 2. RUN
Execute the following two cells to test your predictions.

In [2]:
# Count of transactions by time of day
df.groupby('time_of_day')['transaction_id'].count()

time_of_day
Afternoon    3519
Evening      1021
Morning      5460
Name: transaction_id, dtype: int64

In [3]:
# Average price by time of day
df.groupby('time_of_day')['amount'].mean()

time_of_day
Afternoon    9.372685
Evening      9.277081
Morning      6.841857
Name: amount, dtype: float64

### 3. EXPLAIN
* Did the data match your expectations? 
* The *average price* is higher in the Afternoon than the Morning, but the *count* is lower. What does that tell you about consumer behavior at this shop? (Hint: Think about *what* people buy at 8 AM versus 1 PM).

Yes, the data matched our expectations but the transaction amount was surpricingly close with the evening.

We think people in the morning might be in a rush and get just coffee, and people in the afternoon have time to stay longer in the coffee shop so they might be more likely to buy a sandwich for example.



## Part 3: Multiple Aggregations 

Often, a single metric doesn't tell the whole story. We need to see multiple summaries side-by-side. 

### 1. PREDICT
We are going to look at the `item_type` category. 
* Which item type generates the most *total revenue* (sum)?
* Which item type is sold most *frequently* (count)?

We think that coffee generates the most total revenue because it's very common and it's usually in the higher range for price.

We think donuts might be most frequently sold because people may buy 10 donuts but it's unlikely they'll buy more than 1 coffee for themselves.


### 2. RUN
We can use the `.agg()` method to apply multiple mathematical functions to our groups at the same time.

In [4]:
# Provided Code: Total revenue and items sold per item type
item_summary = df.groupby('item_type').agg({
    'amount': 'sum', 
    'transaction_id': 'count'
})

print(item_summary)

               amount  transaction_id
item_type                            
Coffee       27249.59            4660
Merchandise   8827.82             164
Pastry       12677.88            1971
Sandwich     23766.40            1765
Tea           7289.23            1440


### 3. EXPLAIN
* Why does counting the transaction IDs give us a measure of item popularity?
* Look at the relationship between revenue (first column) and volume (second column). Is your most popular item also your biggest revenue driver? 

Because it shows how many times that item was brought up in the register so it shows us the popularity.

Yes the most popular item also generated the most revenue which makes sense because revenue = price * quantity.

## Part 4: Temporal Analysis

Temporal analysis focuses on understanding how data changes over time.

### 1. PREDICT
We are going to look at the `day_of_week` variable. 
* Do you expect certain days of the week to have higher sales than others? Why?

We expect weekdays to have higher sales than weekends because people are more likley to need coffee for work, however people may have more time to go to a coffee shop during the weekend so it may be close.

We think it'll be close with maybe friday being the highest because people may be more tired from the week and need more coffee.

### 2. RUN


In [5]:
df.groupby('day_of_week').agg({'amount': ['sum', 'count']})

amount      
                  sum count
day_of_week                
Friday       10978.33  1441
Monday       14638.41  1840
Saturday      5715.26   666
Sunday        4667.52   589
Thursday     14877.15  1813
Tuesday      14543.12  1819
Wednesday    14391.13  1832

### 3. EXPLAIN
* What patterns do you notice in sales across different days of the week? 
* What real-world factors not captured in the data might influence this pattern?

Weekdays have a much higher amount of purchases than weekends and weekdays are mostly similar, with friday being slightly lower.

When people go to work is not captured in the data and could reveal a correlation, possibly with causation.


## Part 5: Relationships Between Variables (Covariance)

While aggregating tells us about single columns, **covariance** tells us about the *directional relationship* between two numeric variables. Do they tend to move in the same direction, opposite directions, or is there no pattern at all?

### 1. PREDICT
* Do you expect the covariance between `quantity` and total `amount` to be positive, negative, or close to zero? Why?

Probably positive because the more sales you have the bigger the amount is likely to be.

### 2. RUN
Execute the cell below to calculate the covariance matrix for our two numeric columns.

In [6]:
# Calculate covariance between quantity and total price
df[['quantity', 'amount']].cov()

# The result is a covariance matrix showing the relationship between `quantity` and `amount`. 
# Where each entry represents the covariance between the corresponding pair of variables. 
# The diagonal entries represent the variance of each variable.

,quantity,amount
quantity,0.662372,3.548450
amount,3.548450,82.048197


### 3. EXPLAIN
* Look at the output matrix. The covariance between `quantity` and `amount` is a positive number. What does this confirm about the relationship between the number of items purchased and total cost?
* The resulting number itself might look arbitrary. Unlike *correlation*, covariance is not on a standardized scale of -1 to 1; its raw size depends entirely on the units of the variables (items and dollars). Given this, why might data scientists prefer to present correlation rather than covariance when explaining data relationships to others?
* Using the numbers in the table above, calculate the correlation coefficient between `quantity` and `amount`. 

It confirms that the more items you have in a transaction the higher the price is likely to be.

Data scienctists probably prefer to use correlation instead of covariance because correlation is easier to visualise and interpret, whereas covarience could be large but have no significance in the total amount.

### 4. A FINAL QUESTION
* Let's assume our dataset included a variable for the daily outside temperature and that our coffee shop only sales hot coffee. Would you expect the covariance between `temperature` and the total sales of hot coffee to be positive or negative? Why?

*[Write your answer here]*